# Feature Engineering - SUNT OD Dataset
loads the cleaned parquet from the dataloader, applies feature engineering,
creates the target variable (occupancy_level), and saves X and y ready for model training.

In [3]:
# --- setup: mount google drive and sync repo ---
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_PATH = "/content/drive/MyDrive/Occupancy_capstone/occupancy-prediction-capstone"
%cd {REPO_PATH}

!git config --global user.email "eleonorvilla2003@gmail.com"
!git config --global user.name "victoriaeleonor"

!git pull

Mounted at /content/drive
/content/drive/MyDrive/Occupancy_capstone/occupancy-prediction-capstone
Already up to date.


In [4]:
# --- imports ---
import pandas as pd
import numpy as np
import os
import gc
import pickle
import calendar
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import LabelEncoder

In [5]:
# --- configuration ---

# path to the parquet file generated by the dataloader
FILE_PATH = "/content/drive/MyDrive/Occupancy_capstone/Dataset/sunt_od_202403.parquet"

# where to save processed files
SAVE_PATH = "/content/drive/MyDrive/Occupancy_capstone/Dataset"

# month info (used for output filenames)
YEAR  = 2024
MONTH = 3

# standard bus capacity assumed for occupancy calculation
BUS_CAPACITY = 80

# number of lag features (occupancy at previous stops)
# these are valid features: they capture sequential dependency between stops
N_LAGS = 2

# memory mode controls how many features are created
# 'minimal' : essential features only  (~low RAM)
# 'medium'  : adds trip stage, time of day (~moderate RAM)
# 'full'    : adds aggregations (~high RAM, may crash on colab free)
MEMORY_MODE = 'minimal'

# sampling settings
# IMPORTANT: sampling is applied AFTER lag calculation to preserve stop sequences
# recommended fractions for colab free (12 GB RAM):
#   0.1 = ~2M records  ~1 GB RAM  safe
#   0.3 = ~6M records  ~2 GB RAM  recommended
#   0.5 = ~10M records ~4 GB RAM  use with caution
USE_SAMPLING    = True
SAMPLE_FRACTION = 0.1

# save outputs to drive
SAVE_RESULTS = True

# if true, includes loading_lag_1 and loading_lag_2 as features
# set to false to train a baseline model without sequential context
USE_LAGS = True

print("configuration:")
print(f"  file        : {FILE_PATH}")
print(f"  save path   : {SAVE_PATH}")
print(f"  capacity    : {BUS_CAPACITY} passengers")
print(f"  n lags      : {N_LAGS}")
print(f"  memory mode : {MEMORY_MODE}")
print(f"  sampling    : {SAMPLE_FRACTION*100:.0f}% (applied after lag calculation)")

configuration:
  file        : /content/drive/MyDrive/Occupancy_capstone/Dataset/sunt_od_202403.parquet
  save path   : /content/drive/MyDrive/Occupancy_capstone/Dataset
  capacity    : 80 passengers
  n lags      : 2
  memory mode : minimal
  sampling    : 10% (applied after lag calculation)


In [6]:
# --- load dataset ---

def load_dataset(filepath):
    """
    loads the cleaned parquet file from drive.
    the file should already be sorted by route, direction, trip and stop sequence
    (the dataloader handles this sorting).
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"File not found: {filepath}")

    print(f"Loading: {filepath}")
    df = pd.read_parquet(filepath)

    file_size_mb = os.path.getsize(filepath) / 1024**2
    mem_mb = df.memory_usage(deep=True).sum() / 1024**2

    print(f"  records     : {len(df):,}")
    print(f"  columns     : {list(df.columns)}")
    print(f"  file size   : {file_size_mb:.2f} mb")
    print(f"  memory usage: {mem_mb:.2f} mb")

    return df

df = load_dataset(FILE_PATH)

Loading: /content/drive/MyDrive/Occupancy_capstone/Dataset/sunt_od_202403.parquet
  records     : 19,513,956
  columns     : ['route_short_name', 'register_code', 'direction_id', 'pt_sequence', 'stop_id', 'vehicle', 'trip_number', 'trip_id', 'start_trip', 'end_trip', 'stop_time', 'n-boardings', 'n-alighting', 'lag_loading', 'balance', 'loading']
  file size   : 380.74 mb
  memory usage: 4233.20 mb


In [8]:
# --- feature engineering ---
# IMPORTANT: lags are calculated BEFORE sampling to preserve stop sequences.
# sampling after ensures that lag values are computed from the correct neighboring stops.

def apply_feature_engineering(df, capacity, n_lags, mode='medium'):
    """
    creates temporal, spatial and sequential features.

    parameters:
        df       : cleaned dataframe (already sorted by route/direction/trip/stop)
        capacity : bus capacity in passengers
        n_lags   : number of lag features to create
        mode     : 'minimal', 'medium', or 'full'
    """
    print(f"Applying feature engineering (mode: {mode})...")
    df = df.copy()
    features_created = []

    # -------------------------------------------------------
    # temporal features (all modes)
    # captures daily and weekly patterns in passenger demand
    # -------------------------------------------------------
    print("\n  Temporal features...")
    if 'stop_time' in df.columns:
        df['hour']        = df['stop_time'].dt.hour
        df['day_of_week'] = df['stop_time'].dt.dayofweek
        df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)

        # rush hour: morning (7-9) and evening (17-19)
        df['is_rush_hour'] = (
            ((df['hour'] >= 7) & (df['hour'] <= 9)) |
            ((df['hour'] >= 17) & (df['hour'] <= 19))
        ).astype(int)

        features_created += ['hour', 'day_of_week', 'is_weekend', 'is_rush_hour']
        print(f"  created: {['hour', 'day_of_week', 'is_weekend', 'is_rush_hour']}")
    else:
        print("  skipped (stop_time column not found)")

    # -------------------------------------------------------
    # spatial features (all modes)
    # captures the position of the bus along its route
    # occupancy typically increases early and decreases toward the end
    # -------------------------------------------------------
    print("\n  Spatial features...")
    if 'pt_sequence' in df.columns:
        # relative position of the bus within the route (0-100%)
        df['route_progression'] = df.groupby(
            ['route_short_name', 'direction_id', 'start_trip']
        )['pt_sequence'].transform(lambda x: (x / x.max()) * 100)

        features_created.append('route_progression')
        print(f"  created: ['route_progression']")
    else:
        print("  skipped (pt_sequence column not found)")

    # -------------------------------------------------------
    # sequential (lag) features (all modes)
    # captures occupancy at previous stops within the same trip
    # these are valid features: they use past information, not the current target
    # -------------------------------------------------------
    print(f"\n  Sequential lag features (n_lags={n_lags})...")
    if n_lags > 0 and 'loading' in df.columns:
        for lag in range(1, n_lags + 1):
            col_name = f'loading_lag_{lag}'
            df[col_name] = df.groupby(
                ['route_short_name', 'direction_id', 'start_trip']
            )['loading'].shift(lag)
            features_created.append(col_name)

        print(f"  created: {[f'loading_lag_{i}' for i in range(1, n_lags+1)]}")
    else:
        print("  skipped")

    # -------------------------------------------------------
    # medium and full mode: additional features
    # -------------------------------------------------------
    if mode in ['medium', 'full']:

        # trip stage: start / middle / end of route
        print("\n  Trip stage feature...")
        if 'route_progression' in df.columns:
            df['trip_stage'] = 'middle'
            df.loc[df['route_progression'] <= 25, 'trip_stage'] = 'start'
            df.loc[df['route_progression'] >= 75, 'trip_stage'] = 'end'
            features_created.append('trip_stage')
            print("  created: ['trip_stage']")

        # time of day: night / morning / afternoon / evening
        print("\n Time of day feature...")
        if 'hour' in df.columns:
            df['time_of_day'] = 'afternoon'
            df.loc[df['hour'] < 6,  'time_of_day'] = 'night'
            df.loc[(df['hour'] >= 6)  & (df['hour'] < 12), 'time_of_day'] = 'morning'
            df.loc[df['hour'] >= 18, 'time_of_day'] = 'evening'
            features_created.append('time_of_day')
            print("  created: ['time_of_day']")

    # -------------------------------------------------------
    # full mode: aggregation features (heavy — use with caution)
    # -------------------------------------------------------
    if mode == 'full':
        print("\n  Aggregation features (heavy)...")
        if 'route_short_name' in df.columns:
            route_mean = df.groupby('route_short_name')['loading'].mean().rename('loading_mean_route')
            df = df.join(route_mean, on='route_short_name')
            features_created.append('loading_mean_route')
            print("  created: ['loading_mean_route']")

    # -------------------------------------------------------
    # remove outliers: loading > 120 is physically impossible
    # -------------------------------------------------------
    print("\n  Removing outliers (loading > 120)...")
    n_before = len(df)
    df = df[df['loading'] <= 120].copy()
    removed = n_before - len(df)
    if removed > 0:
        print(f"  removed {removed:,} outlier records")
    else:
        print("  no outliers found")

    # -------------------------------------------------------
    # create target variable: occupancy_level
    # discretized into 4 categories based on % of bus capacity
    #   low       : 0-24%   (0-20 passengers)
    #   medium    : 25-49%  (21-39 passengers)
    #   high      : 50-81%  (40-65 passengers)
    #   very_high : 82%+    (66+ passengers)
    # -------------------------------------------------------
    print("\n  Creating target variable (occupancy_level)...")
    occupancy_pct = (df['loading'] / capacity) * 100
    df['occupancy_level'] = 'medium'
    df.loc[occupancy_pct < 25,  'occupancy_level'] = 'low'
    df.loc[occupancy_pct >= 50, 'occupancy_level'] = 'high'
    df.loc[occupancy_pct >= 82, 'occupancy_level'] = 'very_high'

    print("  Class distribution:")
    dist = df['occupancy_level'].value_counts(normalize=True).sort_index()
    for cls, pct in dist.items():
        print(f"    {cls:12s}: {pct*100:.2f}%")

    # -------------------------------------------------------
    # drop rows with nan in lags or target
    # (first stop of each trip always has nan lags — expected)
    # -------------------------------------------------------
    print("\n  Dropping nan values in lags and target...")
    n_before = len(df)
    lag_cols = [f'loading_lag_{i}' for i in range(1, n_lags + 1) if f'loading_lag_{i}' in df.columns]
    df = df.dropna(subset=['occupancy_level'] + lag_cols)
    removed = n_before - len(df)
    print(f"  removed {removed:,} records with nan ({removed/n_before*100:.2f}%)")

    # -------------------------------------------------------
    # optimize memory: downcast numeric types
    # -------------------------------------------------------
    print("\n  optimizing memory...")
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = df[col].astype('int32')
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype('float32')
    for col in df.select_dtypes(include=['object']).columns:
        if df[col].nunique() < len(df) * 0.5:
            df[col] = df[col].astype('category')

    mem_mb = df.memory_usage(deep=True).sum() / 1024**2
    print(f"Memory after optimization: {mem_mb:.2f} mb")

    print(f"\nFeature engineering complete")
    print(f"  features created : {features_created}")
    print(f"  final shape      : {df.shape}")

    return df

# apply feature engineering BEFORE sampling
n_lags_to_use = N_LAGS if USE_LAGS else 0
df = apply_feature_engineering(df, capacity=BUS_CAPACITY, n_lags=n_lags_to_use, mode=MEMORY_MODE)
gc.collect()

Applying feature engineering (mode: minimal)...

  Temporal features...
  created: ['hour', 'day_of_week', 'is_weekend', 'is_rush_hour']

  Spatial features...
  created: ['route_progression']

  Sequential lag features (n_lags=2)...
  created: ['loading_lag_1', 'loading_lag_2']

  Removing outliers (loading > 120)...
  removed 4,497 outlier records

  Creating target variable (occupancy_level)...
  Class distribution:
    high        : 17.58%
    low         : 46.23%
    medium      : 28.70%
    very_high   : 7.49%

  Dropping nan values in lags and target...
  removed 171,790 records with nan (5.38%)

  optimizing memory...
Memory after optimization: 296.78 mb

Feature engineering complete
  features created : ['hour', 'day_of_week', 'is_weekend', 'is_rush_hour', 'route_progression', 'loading_lag_1', 'loading_lag_2']
  final shape      : (3022798, 24)


68

In [7]:
# --- sampling (applied AFTER feature engineering) ---
# sampling here is safe because lags are already calculated
# from the correct sequential stop order

if USE_SAMPLING:
    n_before = len(df)
    mem_before = df.memory_usage(deep=True).sum() / 1024**2

    df = df.sample(frac=SAMPLE_FRACTION, random_state=42)
    df = df.reset_index(drop=True)
    gc.collect()

    n_after = len(df)
    mem_after = df.memory_usage(deep=True).sum() / 1024**2

    print(f"Sampling applied ({SAMPLE_FRACTION*100:.0f}%):")
    print(f"  records : {n_before:,} -> {n_after:,}")
    print(f"  memory  : {mem_before:.2f} mb -> {mem_after:.2f} mb")
else:
    print("Sampling disabled (USE_SAMPLING=False)")

'''
# --- sample complete trips BEFORE feature engineering ---
# instead of random row sampling, we keep complete trips.
# this way lags calculated within each trip remain valid,
# because all stops of a trip are present in sequence.

if USE_SAMPLING:
    n_before = len(df)
    mem_before = df.memory_usage(deep=True).sum() / 1024**2

    # get all unique trips and sample a fraction of them
    unique_trips = df['trip_number'].unique()
    sampled_trips = pd.Series(unique_trips).sample(frac=SAMPLE_FRACTION, random_state=42)
    df = df[df['trip_number'].isin(sampled_trips)].reset_index(drop=True)
    gc.collect()

    n_after = len(df)
    mem_after = df.memory_usage(deep=True).sum() / 1024**2

    print(f"sampling by complete trips ({SAMPLE_FRACTION*100:.0f}%):")
    print(f"  trips sampled : {len(sampled_trips):,} / {len(unique_trips):,}")
    print(f"  records       : {n_before:,} -> {n_after:,}")
    print(f"  memory        : {mem_before:.2f} mb -> {mem_after:.2f} mb")
else:
    print("sampling disabled")
'''

sampling by complete trips (10%):
  trips sampled : 7 / 66
  records       : 19,513,956 -> 3,199,085
  memory        : 4233.20 mb -> 693.75 mb


In [9]:
# --- prepare for modeling ---
# separates X (features) and y (target)
# removes columns that would cause data leakage

def prepare_for_modeling(df, n_lags):
    """
    separates features (X) and target (y).

    data leakage prevention:
    the following columns are removed because they are directly derived
    from 'loading', which is what the target is based on:
      - loading         : used to compute occupancy_level
      - balance         : n_boardings - n_alighting (derived from loading)
      - lag_loading     : original lag column from the dataset (not our engineered ones)
      - n-boardings     : derived from loading changes
      - n-alighting     : derived from loading changes

    note: loading_lag_1, loading_lag_2 are KEPT because they represent
    occupancy at PREVIOUS stops (past information), not the current stop.
    this is valid sequential context, not leakage.
    """
    y = df['occupancy_level'].copy()

    # columns to remove (leakage or non-informative identifiers)
    cols_to_drop = [
        'occupancy_level',
        # timestamps (not useful as raw values)
        'stop_time', 'start_trip', 'end_trip',
        # identifiers
        'vehicle', 'trip_id', 'register_code', 'trip_number',
        # leakage: directly derived from loading
        'loading',
        'balance',
        'lag_loading',
        'n-boardings',
        'n-alighting',
    ]

    # only drop columns that actually exist
    cols_to_drop = [c for c in cols_to_drop if c in df.columns]
    X = df.drop(columns=cols_to_drop)

    # encode categorical columns
    print("Encoding categorical columns...")
    label_encoders = {}
    for col in X.select_dtypes(include=['object', 'category']).columns:
        label_encoders[col] = LabelEncoder()
        X[col] = label_encoders[col].fit_transform(X[col].astype(str))

    # replace inf and fill remaining nan with median
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median())

    print(f"\nFinal dataset:")
    print(f"  X shape  : {X.shape}")
    print(f"  y shape  : {y.shape}")
    print(f"  features : {list(X.columns)}")
    print(f"\nTarget distribution:")
    dist = y.value_counts(normalize=True).sort_index()
    for cls, pct in dist.items():
        print(f"  {cls:12s}: {pct*100:.2f}%")

    return X, y, label_encoders

X, y, label_encoders = prepare_for_modeling(df, N_LAGS)

Encoding categorical columns...

Final dataset:
  X shape  : (3022798, 11)
  y shape  : (3022798,)
  features : ['route_short_name', 'direction_id', 'pt_sequence', 'stop_id', 'hour', 'day_of_week', 'is_weekend', 'is_rush_hour', 'route_progression', 'loading_lag_1', 'loading_lag_2']

Target distribution:
  high        : 18.28%
  low         : 44.37%
  medium      : 29.53%
  very_high   : 7.82%


In [11]:
# --- save results to drive ---

def save_results(X, y, label_encoders, year, month, save_path):
    """
    saves X, y, and label encoders to drive.
    these files are loaded directly by the training notebooks.
    """
    os.makedirs(save_path, exist_ok=True)
    month_name = calendar.month_name[month].lower()
    suffix = "with_lags" if USE_LAGS else "no_lags"
    prefix = f"{save_path}/sunt_{year}_{month:02d}_{month_name}_{suffix}"

    # save X (features)
    path_X = f"{prefix}_X.parquet"
    X.to_parquet(path_X, index=False)
    print(f"X saved      -> {path_X} ({os.path.getsize(path_X)/1024**2:.2f} mb)")

    # save y (target)
    path_y = f"{prefix}_y.pkl"
    with open(path_y, 'wb') as f:
        pickle.dump(y, f)
    print(f"y saved      -> {path_y} ({os.path.getsize(path_y)/1024**2:.2f} mb)")

    # save label encoders (needed to decode predictions later)
    path_enc = f"{prefix}_encoders.pkl"
    with open(path_enc, 'wb') as f:
        pickle.dump(label_encoders, f)
    print(f"Encoders saved -> {path_enc}")


if SAVE_RESULTS:
    save_results(X, y, label_encoders, YEAR, MONTH, SAVE_PATH)
else:
    print("saving disabled (SAVE_RESULTS=False)")

X saved      -> /content/drive/MyDrive/Occupancy_capstone/Dataset/sunt_2024_03_march_with_lags_X.parquet (21.05 mb)
y saved      -> /content/drive/MyDrive/Occupancy_capstone/Dataset/sunt_2024_03_march_with_lags_y.pkl (49.01 mb)
Encoders saved -> /content/drive/MyDrive/Occupancy_capstone/Dataset/sunt_2024_03_march_with_lags_encoders.pkl


In [12]:
# --- push changes to github ---
!git add .
!git commit -m "fix sampling order and keep lag features"
!git push

[main 6e564a0] fix sampling order and keep lag features
 2 files changed, 1 insertion(+), 2 deletions(-)
 delete mode 100644 Occupancy_dataloader_SUNT_OD (1).ipynb
 rewrite Occupancy_dataloader_SUNT_OD.ipynb (100%)
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 4.83 KiB | 549.00 KiB/s, done.
Total 3 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/victoriaeleonor/occupancy-prediction-capstone.git
   07fdc94..6e564a0  main -> main
